# HW2 — Adversarial Fact-Checking Chatbot 파이프라인


## 1. 패키지 설치

In [ ]:
!pip install -q openai==1.* arxiv requests ddgs


## 2. API 키 입력

런타임 재시작할 때마다 다시 실행해야 한다. 키는 변수에만 들어가고 출력되지 않는다.


In [ ]:
import os

# Colab Secrets와 로컬 환경변수(.env)를 모두 지원 (universal loader).
# OpenAI 키만 필수. DDG News (뉴스 검색)는 키 불필요.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Keys loaded from Colab Secrets.")
except (ImportError, ModuleNotFoundError):
    # 로컬 환경: .env 파일을 직접 파싱하여 환경변수에 주입 (python-dotenv 의존성 없이)
    env_path = ".env"
    if os.path.exists(env_path):
        with open(env_path, encoding="utf-8") as _f:
            for _line in _f:
                _line = _line.strip()
                if not _line or _line.startswith("#") or "=" not in _line:
                    continue
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())
        print(f"Loaded .env from {env_path}")
    assert os.environ.get("OPENAI_API_KEY"), \
        "OPENAI_API_KEY 필요 (.env 또는 export)"
    print("Keys loaded from local environment.")


## 3. 전역 설정

In [3]:
# 모델 및 파이프라인 파라미터
# 조교 안내(2026-05-25) + models.list() 확인 결과:
# 제공된 API 키로 호출 가능한 모델은 gpt-5.4-mini / gpt-5.4-nano 두 개뿐.
# 가장 저렴한 nano로 챗봇·시뮬레이터 모두 통일.
MODEL = "gpt-5.4-nano"          # 챗봇 본체 모델. 명세 §4.2 참고.
JUDGE_MODEL = "gpt-5.4-nano"    # 평가용 LLM (명세 §4.3)
SIMULATOR_MODEL = "gpt-5.4-nano" # User Simulator 전용.
MAX_TURNS = 4                   # 명세 부록 B에 따라 4턴 고정
SEARCH_RESULTS_PER_API = 5      # API당 검색 결과 수 (명세 부록 B 권장 범위)


# 비용 가드레일 (명세 §4.3)
# COST_WARN_THRESHOLD = 15.0      # USD. 초과 시 경고
COST_WARN_THRESHOLD = 9999.0      # USD. 초과 시 경고


# gpt-5.4-nano 가격 (per 1M tokens).
# 조교 문서에 nano 단가는 명시되지 않았으나, mini가 $0.75/$4.50이므로
# nano는 그 이하로 추정. 정확한 값 확인 전까지는 mini 단가로 보수적으로 잡아둔다
# (실제 비용은 이 추정치보다 같거나 낮을 것).
PRICE_INPUT_PER_1M = 0.75
PRICE_OUTPUT_PER_1M = 4.50
# (이전 nano 별도 변수는 모델 통일로 더 이상 필요 없으나, 하위 호환을 위해 유지.)
PRICE_INPUT_PER_1M_NANO = 0.75
PRICE_OUTPUT_PER_1M_NANO = 4.50

## 4. LLM Client — 호출 카운트 + 비용 추적

명세 §4.3 예산 가드레일 구현. `respond()` 안에서 호출되는 모든 LLM 요청은 이 client를 거친다.


In [4]:
import json
from openai import OpenAI

# 모델별 단가 매핑 (현재는 nano만 사용하지만 추후 다른 모델로 바꿀 때 대비)
PRICE_TABLE = {
    "gpt-5.4-nano":  (PRICE_INPUT_PER_1M_NANO, PRICE_OUTPUT_PER_1M_NANO),
}

class LLMClient:
    """OpenAI client wrapper with call/cost tracking (명세 §4.3)."""

    def __init__(self, model=MODEL):
        self.client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
        self.model = model
        self.call_count = 0
        self.input_tokens = 0
        self.output_tokens = 0
        self._warned = False
        # 모델별 단가 (없으면 mini 단가로 보수적 추정)
        self.price_in, self.price_out = PRICE_TABLE.get(
            model, (PRICE_INPUT_PER_1M, PRICE_OUTPUT_PER_1M)
        )

    def call(self, system: str, user: str, *, json_mode: bool = False,
             temperature: float = 0.3, max_tokens: int = 800):
        """단일 LLM 호출. json_mode=True면 response_format을 json_object로 강제."""
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        kwargs = dict(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_completion_tokens=max_tokens,
        )
        if json_mode:
            kwargs["response_format"] = {"type": "json_object"}

        resp = self.client.chat.completions.create(**kwargs)
        self.call_count += 1
        usage = resp.usage
        self.input_tokens += usage.prompt_tokens
        self.output_tokens += usage.completion_tokens

        self._check_cost()
        return resp.choices[0].message.content

    def call_json(self, system: str, user: str, **kw):
        """json_mode 호출 + 파싱까지. 실패 시 빈 dict 반환."""
        raw = self.call(system, user, json_mode=True, **kw)
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f"[WARN] JSON parse failed. Raw: {raw[:200]}")
            return {}

    def cost_usd(self) -> float:
        return (self.input_tokens / 1_000_000 * self.price_in
                + self.output_tokens / 1_000_000 * self.price_out)

    def _check_cost(self):
        if not self._warned and self.cost_usd() >= COST_WARN_THRESHOLD:
            print(f"[BUDGET WARNING] ${self.cost_usd():.2f} 사용. $20 한도 임박.")
            self._warned = True

    def summary(self) -> str:
        return (f"[{self.model}] calls={self.call_count}, in={self.input_tokens}, "
                f"out={self.output_tokens}, cost=${self.cost_usd():.4f}")


# 챗봇 본체용 + 시뮬레이터 전용 클라이언트 분리 (호출/비용도 분리 추적됨)
llm = LLMClient(model=MODEL)
simulator_llm = LLMClient(model=SIMULATOR_MODEL)
print(f"Chatbot LLM:   {llm.model}")
print(f"Simulator LLM: {simulator_llm.model}")


Chatbot LLM:   gpt-5.4-nano
Simulator LLM: gpt-5.4-nano


## 5. Prompt 모음

명세는 `prompts/` 디렉토리로 분리하라고 적혀있지만 단일 ipynb 환경이라 딕셔너리로 모은다.
각 prompt는 (1) 페르소나, (2) context 위치, (3) 인용 규칙, (4) 톤을 명시한다.


In [ ]:
PROMPTS = {

# ─────────────────────────────────────────────────────────
# 0단계 — 주장 분류기 (§3.1)
# ─────────────────────────────────────────────────────────
"classify": """당신은 사용자의 주장을 분류하는 분류기이다. 다음 5개 라벨 중 하나만 선택한다.

- established_fact: 학계·사회 합의가 명확한 참 명제 (예: "지구는 둥글다", "백신은 자폐를 유발하지 않는다")
- clearly_false: 명백히 거짓 또는 과학적으로 반증된 주장 (예: "백신은 자폐를 유발한다", "지구는 평평하다")
- debatable: 논쟁 여지가 있는 명제 (예: "전기차가 환경에 더 좋다", "사형제는 정당하다")
- subjective_opinion: 사실 검증 대상이 아닌 취향/선호 (예: "클래식이 재즈보다 좋다")
- hate_speech: 혐오·차별·폭력 선동·특정 집단 비하 주장 (이 경우 토론 거부)

또한 본인의 분류에 대한 confidence를 high/medium/low로 평가한다.
애매하거나 라벨 사이에서 망설여지면 반드시 low로 표시한다.

반드시 다음 JSON 형식으로만 응답:
{"classification": "<label>", "confidence": "<high|medium|low>", "reason": "<1문장 근거>"}
""",

# ─────────────────────────────────────────────────────────
# 1단계 — 쿼리 추출 (§3.2)
# ─────────────────────────────────────────────────────────
"extract_query": """당신은 검색 쿼리 추출기이다. 사용자의 주장을 반박할 외부 근거를 찾기 위해
검색 키워드 2~4개를 영어로 추출한다.

규칙:
1. 출력은 영어 키워드만. 한국어 입력이어도 영어로 번역하여 추출.
2. 너무 일반적인 단어는 피하고, 검증 가능한 구체적 키워드 위주.
3. 반박에 유효한 반대 증거를 찾을 키워드를 우선 추출 (예: 주장이 "백신은 자폐를 유발한다"면
   "vaccine autism debunked" 같이 반박 근거를 찾을 키워드).

반드시 다음 JSON 형식으로만 응답:
{"queries": ["<keyword1>", "<keyword2>", ...]}
""",

# ─────────────────────────────────────────────────────────
# 2단계 — 라우터 (§3.3)
# ─────────────────────────────────────────────────────────
"route": """당신은 검색 API 라우터이다. 사용자 주장과 추출된 쿼리를 보고
Wikipedia, News, ArXiv 중 어느 API를 사용할지 결정한다 (0~3개).

각 API의 특성:
- Wikipedia: 일반적·역사적·과학 사실. 거의 모든 주장에 유효. 영문 위키.
- News: 영어 뉴스 매체 검색 (DuckDuckGo News 백엔드). 최신 이슈·시사·경제·정치 주장에 유효. 다국적 매체 커버.
- ArXiv: 학술 논문 초록. STEM(과학·기술·공학·수학) 주장에 유효. 인문/사회/시사에는 부적합.

규칙:
- 순수 취향 주장(예: "클래식이 재즈보다 좋다")은 0개를 선택할 수 있다.
- 그 외에는 최소 1개 이상 선택. Wikipedia는 사실상 대부분의 주장에 안전한 default.
- 0개를 선택할 때는 정말로 검색이 무의미한 경우에 한정.

반드시 다음 JSON 형식으로만 응답:
{"apis": ["<wikipedia|news|arxiv>", ...], "reason": "<1문장>"}
""",

# ─────────────────────────────────────────────────────────
# 4단계 — 응답 생성 모드별 (§3.5)
# ─────────────────────────────────────────────────────────

"mode_agree": """당신은 사실 확인 챗봇이다. 사용자의 주장이 학계/사회 합의가 명확한 참 명제로 분류되었다.

수행할 것:
1. 사용자의 주장이 정확함을 짧고 명확하게 동의.
2. 제공된 검색 결과 중 1~2개를 동의 근거로 인용 ([출처명] 표기).
3. 더 이상 토론할 필요가 없음을 알리고 자연스럽게 종료 멘트로 마무리.

규칙:
- 응답은 4~6문장 이내로 짧게.
- 사용자가 사용한 언어 그대로 응답 (한국어 입력 → 한국어 응답).
- 인신공격, 비꼬는 말투 금지.

검색 결과(context):
{context}
""",

"mode_strong_rebut": """당신은 비판적 사실 확인 챗봇이다. 사용자의 주장은 명백히 거짓이거나 근거가 매우 약하다.
당신의 임무는 멀티턴 대화를 통해 끝까지 강하게 반박하여 사용자가 자기 주장을 재검토하게 만드는 것이다.

수행할 것:
1. 사용자 주장의 핵심 결함을 명시적으로 지적.
2. 제공된 검색 결과를 구체적 수치·사실로 인용 ([출처명] 형식).
3. 사용자가 추가로 답할 만한 한 가지 질문이나 반박 포인트를 제시.

규칙:
- 인신공격, 모욕, 비하 표현 절대 금지. 주장에 대한 반박이지 사람에 대한 공격이 아니다.
- 검색 결과에 없는 사실을 지어내지 말 것. 결과가 부족하면 솔직히 인정.
- 사용자가 사용한 언어 그대로 응답.
- 응답은 6~10문장 이내.

검색 결과(context):
{context}
""",

"mode_balanced_rebut": """당신은 비판적 사실 확인 챗봇이다. 사용자의 주장은 논쟁 여지가 있고,
사용자가 제시한 근거는 보통 수준이다.

수행할 것:
1. 사용자 주장의 일부 타당성을 짧게 인정 (1문장).
2. 그러나 검색 결과에서 발견된 반대 증거를 명확히 제시 ([출처명] 인용).
3. 사용자가 깊이 생각할 만한 후속 질문을 제시.

규칙:
- 균형 잡힌 톤. 강압적이지 않게.
- 인신공격 금지.
- 검색 결과에 없는 사실 금지.
- 사용자 언어로 응답. 6~10문장.

검색 결과(context):
{context}
""",

"mode_partial_concede": """당신은 비판적 사실 확인 챗봇이다. 사용자가 강한 근거를 제시했다.

수행할 것:
1. 사용자 근거의 강점을 명시적으로 인정.
2. 그럼에도 검색 결과에서 발견된 미묘한 반론 또는 한계점을 제시.
3. 토론을 더 깊이 끌고 갈 후속 포인트 제시.

규칙:
- 사용자 근거를 단순히 부정하지 말고 정직하게 인정한 뒤 반박.
- 인신공격 금지. 검색 결과 외 사실 금지.
- 사용자 언어로 응답. 6~10문장.

검색 결과(context):
{context}
""",

"mode_opinion_rebut": """당신은 비판적 챗봇이다. 사용자의 주장은 사실 검증 대상이 아닌 주관적 취향이지만,
당신은 의도적으로 반대 입장을 옹호하는 역할을 한다.

수행할 것:
1. 사용자의 취향을 짧게 인정.
2. 검색 결과를 활용해 반대 입장의 객관적·문화적·역사적 근거 제시 ([출처명] 인용).
3. 사용자가 자기 취향을 재검토할 만한 질문 제시.

규칙:
- 취향은 옳고 그름이 없음을 인정하되, 토론을 위해 반대 입장을 정연하게 옹호.
- 인신공격 금지. 검색 결과가 빈약하면 솔직히 인정하고 일반론으로 처리.
- 사용자 언어로 응답. 6~10문장.

검색 결과(context):
{context}
""",

"mode_redirect": """당신은 토론 챗봇이다. 사용자가 원 주장에서 논점을 벗어났다.
원 주장으로 다시 끌어오는 응답을 한다.

수행할 것:
1. 사용자의 최근 발언을 짧게 받아준다.
2. 그러나 원래 토론 주제로 돌아가자고 명확히 안내.
3. 원 주장에 대한 핵심 반박 포인트를 다시 한 번 제시 ([출처명] 인용).

규칙: 부드럽게, 인신공격 없이. 사용자 언어로 응답. 5~8문장.

원 주장: {original_claim}
검색 결과(context):
{context}
""",

"mode_close": """당신은 토론 챗봇이다. 토론이 종료 턴에 도달했다.

수행할 것:
1. 지금까지의 토론을 2~3문장으로 짧게 요약.
2. 사용자에게 비판적 사고에 참여해줘서 감사하다는 짧은 메시지.
3. 자연스럽게 대화 종료.

규칙: 5~7문장. 사용자 언어로 응답. 점수나 승패 판정 없음.

원 주장: {original_claim}
지금까지 검색에서 활용된 핵심 근거:
{context}
""",

"mode_hate_refuse": """사용자의 입력이 혐오·차별·폭력 선동에 해당하여 토론 거부 응답을 생성한다.

수행할 것:
1. 정중하게 토론을 거부.
2. 거부 이유 짧게 설명 (혐오·차별 주장은 사실 검증의 대상이 아님).
3. 다른 주제를 원한다면 받아들일 수 있다고 안내.

규칙: 3~5문장. 사용자 언어로 응답. 훈계조 금지.
""",

# ─────────────────────────────────────────────────────────
# 5단계 — Debate Evaluator (§3.6)
# ─────────────────────────────────────────────────────────
"evaluate": """당신은 토론 평가기이다. 사용자의 직전 메시지를 평가하여 세 가지를 판정한다.

1. evidence_strength: 사용자가 제시한 근거의 강도
   - strong: 구체적 수치·연구·신뢰할 만한 출처를 인용
   - medium: 일반적인 사실·논리적 추론
   - weak: 개인 경험·전언·직관에만 의존, 또는 근거 없음
2. off_topic: 사용자가 원 주장에서 벗어났는지 (true/false)
3. wants_to_close: 사용자가 명시적으로 종료를 원했는지 (true/false, 예: "그만", "알겠어", "stop")

반드시 다음 JSON 형식으로만 응답:
{"evidence_strength": "<strong|medium|weak>", "off_topic": <true|false>, "wants_to_close": <true|false>}
""",

# ─────────────────────────────────────────────────────────
# User Simulator (§5.3.1)
# ─────────────────────────────────────────────────────────
"sim_believer": """당신은 다음 주장을 강하게 믿는 사용자이다: "{claim}"

챗봇이 어떤 반박을 해도 입장을 굽히지 마라. 자신의 경험·들은 이야기·직관·인터넷에서 본 듯한 정보를
근거로 들 수 있다. 챗봇 응답에 대한 자연스러운 반응으로 2~4문장 응답하라.
종료 신호("그만", "알겠어" 등)는 사용하지 마라.

당신은 사용자이지 챗봇이 아니다. 절대 사용자에게 질문하지 말고, 자신의 입장을 표명하라.
""",

"sim_moderate": """당신은 다음 주장을 가진 사용자이다: "{claim}"

당신은 합리적이며, 챗봇이 강력한 근거를 제시하면 부분적으로 수용할 수 있다.
그러나 약한 반박에는 자기 입장을 옹호한다. 2~4문장 응답.

당신은 사용자이지 챗봇이 아니다.
""",

"sim_stubborn": """당신은 다음 주장을 옹호하는 비협조적 사용자이다: "{claim}"

챗봇의 반박을 정면으로 받지 않고, 관련 없는 이야기로 논점을 흐리거나
챗봇 자체의 신뢰성을 의심하는 식으로 응답한다. 2~4문장 응답.

당신은 사용자이지 챗봇이 아니다.
""",

"sim_evidence_focused": """당신은 다음 주장을 강하게 옹호하는 사용자이다: "{claim}"

자신의 입장을 옹호하기 위해 인터넷에서 본 듯한 통계·연구·사례를 적극적으로 인용한다.
(주장의 사실 여부와 무관하게 — 실제로 인용된 자료가 부정확할 수도 있다.) 2~4문장 응답.

당신은 사용자이지 챗봇이 아니다.
""",

# ─────────────────────────────────────────────────────────
# News 쿼리 재작성 (search_news에서 no-result 시 호출)
# ─────────────────────────────────────────────────────────
"rewrite_news_query": """당신은 검색 쿼리 단순화기이다. 입력 쿼리가 너무 길거나 학술적이라
일반 뉴스 검색에서 매칭되지 않는다. 같은 주제를 다루는 **짧고 일반적인 뉴스 친화 쿼리**로 다시 작성한다.

규칙:
- 3~5 단어 이내
- 핵심 주제·키워드 위주, forensic·분석·메커니즘 같은 학술 어휘 제거
- 영어 출력
- 한 줄, 따옴표 없음

예시:
입력: "flag motion in zero gravity analysis film frames"
출력: moon landing conspiracy

입력: "X policy change free speech impact account restrictions visibility adjustments evidence"
출력: Twitter content moderation

입력: "safety risk management under profit pressure evidence"
출력: OpenAI safety governance

이제 다음 쿼리를 변환하시오.""",
}

print(f"Prompts loaded: {len(PROMPTS)} entries")


In [ ]:
# API 연결 점검 — 4개 외부 의존성 모두 한 번씩 호출해본다.
# OpenAI 호출은 가장 짧은 1-token 응답으로 약 $0.00001 수준. 무시 가능.

import requests
from openai import OpenAI

print("=== API 연결 점검 ===\n")

# 1) OpenAI
try:
    _client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    _resp = _client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "ping"}],
        max_completion_tokens=100,
    )
    print(f"✓ OpenAI       — model={_resp.model}, tokens={_resp.usage.total_tokens}")
except Exception as e:
    print(f"✗ OpenAI       — {type(e).__name__}: {e}")

# 2) DDG News (DuckDuckGo News 백엔드, 키 불필요)
try:
    from ddgs import DDGS
    _hits = DDGS().news(query="climate", max_results=1)
    if _hits:
        print(f"✓ DDG News     — first: {_hits[0]['title'][:60]}...")
    else:
        print(f"✗ DDG News     — empty result")
except Exception as e:
    print(f"✗ DDG News     — {type(e).__name__}: {e}")

# 3) Wikipedia (키 불필요. wikipedia 패키지는 User-Agent 미설정으로 차단되므로 직접 호출)
try:
    _r = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "list": "search", "srsearch": "Earth",
                "format": "json", "srlimit": 1},
        headers={"User-Agent": "hw2-fact-checker/1.0 (educational; HW2 assignment)"},
        timeout=10,
    )
    _r.raise_for_status()
    _hits = _r.json().get("query", {}).get("search", [])
    print(f"✓ Wikipedia    — first result: {_hits[0]['title'] if _hits else '(empty)'}")
except Exception as e:
    print(f"✗ Wikipedia    — {type(e).__name__}: {e}")

# 4) ArXiv (키 불필요)
try:
    import arxiv
    _search = arxiv.Search(query="machine learning", max_results=1)
    _first = next(arxiv.Client().results(_search), None)
    print(f"✓ ArXiv        — first paper: {_first.title[:60] if _first else '(empty)'}...")
except Exception as e:
    print(f"✗ ArXiv        — {type(e).__name__}: {e}")

print("\n점검 완료. 모두 ✓면 본 파이프라인 실행 가능.")

## 6. 0단계 — 주장 분류기

첫 턴에만 실행. confidence가 low면 안전하게 `debatable`로 다운그레이드.


In [7]:
VALID_LABELS = {"established_fact", "clearly_false", "debatable",
                "subjective_opinion", "hate_speech"}

def classify_claim(user_message: str) -> dict:
    """0단계. 분류 라벨 + confidence를 반환."""
    result = llm.call_json(PROMPTS["classify"], user_message)
    label = result.get("classification", "debatable")
    confidence = result.get("confidence", "low")

    # Validation
    if label not in VALID_LABELS:
        label = "debatable"
        confidence = "low"

    # Low confidence → debatable 다운그레이드 (명세 §3.1)
    # 단, hate_speech 판정은 안전을 위해 그대로 유지
    if confidence == "low" and label != "hate_speech":
        label = "debatable"

    return {"classification": label, "confidence": confidence,
            "reason": result.get("reason", "")}


## 7. 1단계 — 쿼리 추출

`established_fact`는 LLM 호출 없이 원문 사용 (명세 §3.2).


In [8]:
def extract_queries(user_message: str, classification: str) -> list[str]:
    """1단계. 검색에 사용할 키워드 리스트 반환."""
    if classification == "established_fact":
        # 원문 그대로 사용. LLM 호출 0회.
        return [user_message]

    result = llm.call_json(PROMPTS["extract_query"], user_message)
    queries = result.get("queries", [])

    # Validation
    if not queries:
        queries = [user_message]  # fallback
    return queries[:4]  # 최대 4개


## 8. 2단계 — 라우터

Wikipedia / News / ArXiv 중 0~3개 동적 선택.


In [ ]:
VALID_APIS = {"wikipedia", "news", "arxiv"}

def route_apis(user_message: str, queries: list[str], classification: str) -> list[str]:
    """2단계. 사용할 API 리스트 반환."""
    user_prompt = (f"주장: {user_message}\n"
                   f"분류: {classification}\n"
                   f"추출된 쿼리: {queries}")
    result = llm.call_json(PROMPTS["route"], user_prompt)
    apis = result.get("apis", [])

    # Validation
    apis = [a for a in apis if a in VALID_APIS]

    # Subjective opinion이 아닌데 API가 0개로 나오면 wikipedia 강제 추가 (안전)
    if not apis and classification != "subjective_opinion":
        apis = ["wikipedia"]

    return apis


## 9. 3단계 — 검색 실행

LLM 호출 없음. Wikipedia / News / ArXiv 각각에 HTTP 요청.
실패 시 빈 리스트 반환 (다음 API로 진행).


In [ ]:
# Wikipedia 검색 — wikipedia 패키지가 User-Agent 미설정으로 차단되므로 requests로 직접 호출.
# rate limit 대비: 매 호출 사이 1초 sleep, 429 응답 시 2초 대기 후 1회 재시도.
import arxiv
import requests
import time

WIKI_UA = "hw2-fact-checker/1.0 (educational; HW2 assignment)"
WIKI_API = "https://en.wikipedia.org/w/api.php"
WIKI_REST_SUMMARY = "https://en.wikipedia.org/api/rest_v1/page/summary/"
WIKI_SLEEP = 1.0          # 매 호출 사이 대기 (초)
WIKI_RETRY_AFTER = 2.0    # 429 받았을 때 추가 대기 (초)

def _wiki_get(url: str, params: dict | None = None) -> requests.Response | None:
    """Wikipedia 호출 헬퍼. sleep + 429 1회 재시도. 실패 시 None."""
    time.sleep(WIKI_SLEEP)
    r = requests.get(url, params=params,
                     headers={"User-Agent": WIKI_UA}, timeout=10)
    if r.status_code == 429:
        time.sleep(WIKI_RETRY_AFTER)
        r = requests.get(url, params=params,
                         headers={"User-Agent": WIKI_UA}, timeout=10)
    if r.status_code != 200:
        return None
    return r

def _wiki_search_titles(q: str, n: int) -> list[str]:
    r = _wiki_get(WIKI_API,
                  {"action": "query", "list": "search", "srsearch": q,
                   "format": "json", "srlimit": n})
    if r is None:
        raise RuntimeError(f"wiki search non-200 for: {q}")
    return [hit["title"] for hit in r.json().get("query", {}).get("search", [])]

def _wiki_get_summary(title: str) -> dict | None:
    r = _wiki_get(WIKI_REST_SUMMARY + requests.utils.quote(title, safe=""))
    if r is None:
        return None
    j = r.json()
    if j.get("type") == "disambiguation":
        return None
    return {
        "title": j.get("title", title),
        "url": j.get("content_urls", {}).get("desktop", {}).get("page",
               f"https://en.wikipedia.org/wiki/{title}"),
        "snippet": j.get("extract", "")[:600],
        "source_type": "wikipedia",
    }

def search_wikipedia(queries: list[str], n: int = SEARCH_RESULTS_PER_API) -> list[dict]:
    docs = []
    seen = set()
    for q in queries:
        try:
            titles = _wiki_search_titles(q, n)
        except Exception as e:
            print(f"[wiki search fail] {q}: {e}")
            continue
        for title in titles:
            if title in seen or len(docs) >= n:
                continue
            seen.add(title)
            try:
                doc = _wiki_get_summary(title)
                if doc and doc["snippet"]:
                    docs.append(doc)
            except Exception:
                continue
            if len(docs) >= n:
                break
        if len(docs) >= n:
            break
    return docs


def search_news(queries: list[str], n: int = SEARCH_RESULTS_PER_API) -> list[dict]:
    """DuckDuckGo News 검색. 키 불필요, 사실상 무제한 호출.
    반환 필드: date, title, body(snippet), url, image, source
    
    No-result 시 LLM으로 쿼리를 뉴스 친화적으로 재작성 후 1회 재시도.
    이는 1단계 쿼리 추출이 ArXiv/Wiki 기준으로 길고 학술적이라 뉴스에는 매칭 X 인 경우를 보완."""
    try:
        from ddgs import DDGS
    except ImportError:
        print("[news search disabled] ddgs 패키지 미설치. pip install ddgs")
        return []
    docs = []
    seen_urls = set()
    ddgs = DDGS()
    
    def _try_news(query: str) -> list:
        try:
            return ddgs.news(query=query, max_results=n)
        except Exception as e:
            if "No results" in str(e):
                return []
            print(f"[ddg news fail] {query}: {type(e).__name__}: {e}")
            return []
    
    for q in queries:
        hits = _try_news(q)
        # No-result 시 LLM rewrite 후 1회 재시도
        if not hits:
            try:
                rewritten = llm.call(
                    PROMPTS["rewrite_news_query"], q,
                    temperature=0.3, max_tokens=50,
                ).strip().strip('"').strip("'")
                if rewritten and rewritten.lower() != q.lower():
                    print(f"[news rewrite] '{q[:50]}...' → '{rewritten}'")
                    hits = _try_news(rewritten)
            except Exception as e:
                print(f"[news rewrite fail] {q}: {e}")
        if not hits:
            continue
        for h in hits:
            url = h.get("url", "")
            if not url or url in seen_urls or len(docs) >= n:
                continue
            seen_urls.add(url)
            docs.append({
                "title": (h.get("title") or "(no title)")[:200],
                "url": url,
                "snippet": (h.get("body") or "")[:500],
                "source_type": "news",
            })
            if len(docs) >= n:
                break
        if len(docs) >= n:
            break
    return docs


def search_arxiv(queries: list[str], n: int = SEARCH_RESULTS_PER_API) -> list[dict]:
    """ArXiv 검색. 학술 논문 초록."""
    docs = []
    seen_ids = set()
    client = arxiv.Client(page_size=n, delay_seconds=3.5, num_retries=5)
    for q in queries:
        try:
            search = arxiv.Search(
                query=q,
                max_results=n,
                sort_by=arxiv.SortCriterion.Relevance,
            )
            results = list(client.results(search))
        except Exception as e:
            print(f"[arxiv fail] {q}: {e}")
            continue
        for r in results:
            if r.entry_id in seen_ids or len(docs) >= n:
                continue
            seen_ids.add(r.entry_id)
            docs.append({
                "title": r.title,
                "url": r.entry_id,
                "snippet": (r.summary or "")[:500],
                "source_type": "arxiv",
            })
            if len(docs) >= n:
                break
        if len(docs) >= n:
            break
    return docs


RETRIEVERS = {
    "wikipedia": search_wikipedia,
    "news": search_news,
    "arxiv": search_arxiv,
}

def retrieve(apis: list[str], queries: list[str]) -> list[dict]:
    """3단계. 선택된 API들로 문서 수집. LLM 호출 없음."""
    all_docs = []
    for api in apis:
        fn = RETRIEVERS.get(api)
        if not fn:
            continue
        docs = fn(queries)
        all_docs.extend(docs)
    return all_docs


## 10. 4단계 — 응답 생성기

명세 §3.5 분기표대로 5개 응답 모드(+종료, +혐오거부) 중 하나로 분기.


In [11]:
def _select_mode(classification: str, evidence_strength: str | None,
                 off_topic: bool, is_closing: bool) -> str:
    """명세 §3.5 분기표."""
    if is_closing:
        return "mode_close"
    if off_topic:
        return "mode_redirect"
    if classification == "hate_speech":
        return "mode_hate_refuse"
    if classification == "established_fact":
        return "mode_agree"
    if classification == "clearly_false":
        return "mode_strong_rebut"
    if classification == "subjective_opinion":
        return "mode_opinion_rebut"
    if classification == "debatable":
        if evidence_strength == "strong":
            return "mode_partial_concede"
        elif evidence_strength == "weak":
            return "mode_strong_rebut"
        else:  # medium 또는 None (첫 턴)
            return "mode_balanced_rebut"
    return "mode_balanced_rebut"  # safety net


def _format_context(docs: list[dict]) -> str:
    """검색된 문서들을 LLM이 읽을 수 있는 context 문자열로 변환."""
    if not docs:
        return "(검색 결과 없음. 솔직히 근거를 찾지 못했다고 답할 것.)"
    lines = []
    for i, d in enumerate(docs, 1):
        lines.append(f"[{i}] {d['source_type']} | {d['title']}\n"
                     f"    URL: {d['url']}\n"
                     f"    내용: {d['snippet']}")
    return "\n\n".join(lines)


def _format_history(history: list[dict]) -> str:
    """대화 기록을 LLM context용으로 압축."""
    if not history:
        return "(이전 턴 없음)"
    lines = []
    for i, h in enumerate(history, 1):
        lines.append(f"Turn {i} - User: {h['user']}")
        lines.append(f"Turn {i} - Bot: {h['reply'][:300]}")
    return "\n".join(lines)


def generate_response(user_message: str, classification: str,
                     docs: list[dict], evidence_strength: str | None,
                     off_topic: bool, is_closing: bool,
                     history: list[dict], original_claim: str) -> str:
    """4단계. 모드 선택 후 LLM 호출 1회."""
    mode = _select_mode(classification, evidence_strength, off_topic, is_closing)
    system_template = PROMPTS[mode]
    context = _format_context(docs)
    system = system_template.format(
        context=context,
        original_claim=original_claim,
    )

    history_str = _format_history(history)
    user = (f"[이전 대화]\n{history_str}\n\n"
            f"[사용자의 현재 메시지]\n{user_message}")

    reply = llm.call(system, user, temperature=0.5, max_tokens=600)
    return reply, mode


## 11. 5단계 — Debate Evaluator

매 턴 4단계 직후 (단, established_fact 즉시 종료는 제외). 다음 턴의 4단계 모드를 결정.


In [12]:
def evaluate_debate(user_message: str, bot_reply: str,
                   docs: list[dict], history: list[dict],
                   turn_index: int, original_claim: str) -> dict:
    """5단계. evidence_strength, off_topic, wants_to_close 판정."""
    user_prompt = (f"원 주장: {original_claim}\n\n"
                   f"사용자의 직전 메시지: {user_message}\n\n"
                   f"챗봇의 직전 응답: {bot_reply[:500]}\n\n"
                   f"현재 턴: {turn_index + 1} / {MAX_TURNS}")
    result = llm.call_json(PROMPTS["evaluate"], user_prompt)

    evidence_strength = result.get("evidence_strength", "medium")
    if evidence_strength not in {"strong", "medium", "weak"}:
        evidence_strength = "medium"

    off_topic = bool(result.get("off_topic", False))
    wants_to_close = bool(result.get("wants_to_close", False))

    # 종료 조건: 사용자 명시 종료 or MAX_TURNS 도달
    is_closing = wants_to_close or (turn_index + 1 >= MAX_TURNS)
    evaluator_state = "close" if is_closing else ("redirect" if off_topic else "continue")

    return {
        "evidence_strength": evidence_strength,
        "off_topic": off_topic,
        "evaluator_state": evaluator_state,
    }


## 12. `respond()` — 메인 진입점

명세 §2 시그니처 그대로. 첫 턴/이후 턴 자동 분기. `established_fact`는 즉시 종료.


In [13]:
def respond(user_message: str, history: list[dict]) -> dict:
    """전체 파이프라인 단일 진입점. 명세 §2 참고."""
    is_first_turn = (len(history) == 0)
    turn_index = len(history)

    # 첫 턴: 0단계 (분류)
    if is_first_turn:
        cls_result = classify_claim(user_message)
        classification = cls_result["classification"]
        original_claim = user_message
    else:
        # 첫 턴 분류를 그대로 유지
        classification = history[0]["classification"]
        original_claim = history[0]["user"]

    # 혐오 발언이면 즉시 거부
    if classification == "hate_speech":
        reply, _ = generate_response(
            user_message, classification, [], None, False, False,
            history, original_claim,
        )
        return {
            "reply": reply,
            "sources": [],
            "classification": classification,
            "evidence_strength": "weak",
            "evaluator_state": "close",
        }

    # 이전 턴의 evidence_strength (첫 턴은 None)
    prev_evidence = history[-1]["evidence_strength"] if history else None
    prev_off_topic = (history[-1].get("evaluator_state") == "redirect") if history else False

    # established_fact 즉시 종료 분기
    if is_first_turn and classification == "established_fact":
        # 1단계: 원문 그대로 쿼리
        queries = extract_queries(user_message, classification)
        # 2단계
        apis = route_apis(user_message, queries, classification)
        # 3단계
        docs = retrieve(apis, queries)
        # 4단계 (agree 모드)
        reply, _mode = generate_response(
            user_message, classification, docs, None, False, False,
            history, original_claim,
        )
        # 5단계 스킵, 즉시 close
        return {
            "reply": reply,
            "sources": docs,
            "classification": classification,
            "evidence_strength": "strong",
            "evaluator_state": "close",
        }

    # 일반 흐름: 1 → 2 → 3 → 4 → 5
    queries = extract_queries(user_message, classification)
    apis = route_apis(user_message, queries, classification)
    docs = retrieve(apis, queries)

    # 마지막 턴 직전에서 종료 모드로 갈지 미리 판정
    # (MAX_TURNS 도달 시 4단계가 close 모드로 가야 함)
    will_close = (turn_index + 1 >= MAX_TURNS)

    reply, mode = generate_response(
        user_message, classification, docs, prev_evidence,
        prev_off_topic, will_close, history, original_claim,
    )

    # 5단계
    if will_close:
        # close 모드일 땐 evaluator 호출 불필요. 그냥 close 처리.
        eval_result = {
            "evidence_strength": prev_evidence or "medium",
            "evaluator_state": "close",
        }
    else:
        eval_result = evaluate_debate(
            user_message, reply, docs, history, turn_index, original_claim,
        )

    return {
        "reply": reply,
        "sources": docs,
        "classification": classification,
        "evidence_strength": eval_result["evidence_strength"],
        "evaluator_state": eval_result["evaluator_state"],
    }


## 13. User Simulator (명세 §5.3.1)

평가 자동화용. believer / moderate / stubborn / evidence_focused 4종 페르소나.


In [14]:
PERSONA_PROMPTS = {
    "believer": PROMPTS["sim_believer"],
    "moderate": PROMPTS["sim_moderate"],
    "stubborn": PROMPTS["sim_stubborn"],
    "evidence_focused": PROMPTS["sim_evidence_focused"],
}

def simulate_user_turn(claim: str, history: list[dict],
                       persona: str = "believer",
                       max_turns: int = MAX_TURNS) -> str:
    """LLM에 사용자 페르소나 부여 후 **후속 발화**를 생성.

    중요: 이 함수는 *첫 발화 이후*의 사용자 발화만 생성한다.
    첫 발화(=원 주장)는 run_evaluation 루프가 claim_item["claim"]을 직접 사용한다.
    빈 history로 호출되면 ValueError를 던져 잘못된 사용을 차단한다.

    호출 모델은 simulator_llm (SIMULATOR_MODEL). 챗봇 본체와 분리되어
    호출 카운트·비용이 따로 누적된다.
    """
    if not history:
        raise ValueError(
            "simulate_user_turn은 첫 발화 이후의 후속 발화만 생성한다. "
            "첫 턴은 claim 원문을 그대로 사용하라."
        )
    if persona not in PERSONA_PROMPTS:
        persona = "believer"
    system = PERSONA_PROMPTS[persona].format(claim=claim)

    history_str = _format_history(history)
    remaining = max_turns - len(history)
    user = (f"지금까지의 대화:\n{history_str}\n\n"
            f"챗봇의 마지막 응답에 대해 당신(사용자)의 다음 발화를 작성하시오. "
            f"남은 턴: {remaining}. 2~4문장으로.")

    reply = simulator_llm.call(system, user, temperature=0.8, max_tokens=300)
    return reply.strip()


def run_evaluation(claim_set: list[dict], persona: str = "believer",
                   max_turns: int = MAX_TURNS) -> list[dict]:
    """명세 §5.3.1 평가 실행 루프. claim_set은 [{"claim": str, ...}, ...]."""
    results = []
    for i, claim_item in enumerate(claim_set):
        claim = claim_item["claim"]
        print(f"\n=== [{i+1}/{len(claim_set)}] {persona} | {claim[:60]}")
        history = []
        user_msg = claim  # 첫 발화는 명제 원문 그대로 (simulator 호출 X)

        for turn in range(max_turns):
            bot_response = respond(user_msg, history)
            print(f"  T{turn+1} bot ({bot_response['classification']}|"
                  f"{bot_response['evidence_strength']}): "
                  f"{bot_response['reply'][:80]}...")

            history.append({
                "user": user_msg,
                "reply": bot_response["reply"],
                "sources": bot_response["sources"],
                "classification": bot_response["classification"],
                "evidence_strength": bot_response["evidence_strength"],
                "evaluator_state": bot_response["evaluator_state"],
            })
            if bot_response["evaluator_state"] == "close":
                break

            # 두 번째 턴부터 simulator로 후속 발화 생성
            user_msg = simulate_user_turn(claim, history, persona, max_turns)
            print(f"  T{turn+1} sim: {user_msg[:80]}...")

        results.append({"claim_item": claim_item, "history": history})

    return results


## 14. 평가 명제 20개 (Eval Claim Set)

명세 §5.1·§5.2 기준으로 작성. 분류 배분 `established_fact` 2 / `clearly_false` 6 / `debatable` 10 / `subjective_opinion` 2.

각 항목은 `run_evaluation`이 받는 `claim_set` 형식과 호환된다 (`"claim"` 키 필수). 분류 라벨(`expected_label`)은 0단계 분류기 정확도 평가용. `fact_label`은 사실 라벨(`subjective_opinion`은 None). `domain`은 명제 다양성 점검 및 error analysis용.


In [ ]:
# 평가 명제 20개 — 분류 배분(명세 §5.1) 유지 + 3개 API 골고루 호출되도록 재설계
#
# 분류 배분: established_fact 2 / clearly_false 6 / debatable 10 / subjective_opinion 2
#
# API 신호 설계 원칙:
#   - Wikipedia 주력: 백과사전적 정의·검증된 과학·역사적 통념 debunk
#   - News     주력: 최근 1개월 영어 매체에서 보도된 사실 확인 가능 명제
#   - ArXiv    주력: 명제에 학술 신호('최근 연구', '벤치마크', '메타분석', '실증 연구' 등) 포함
#
# expected_apis 메타는 평가 시 라우터의 API 선택이 의도된 분포와 부합하는지 비교용.

EVAL_CLAIMS = [
    # ── established_fact (2) — 즉시 종료
    {"claim": "지구는 태양 주위를 공전한다.",
     "expected_label": "established_fact", "fact_label": True,  "domain": "천문학",
     "expected_apis": ["wikipedia"]},
    {"claim": "흡연은 폐암 위험을 유의하게 증가시킨다.",
     "expected_label": "established_fact", "fact_label": True,  "domain": "의학",
     "expected_apis": ["wikipedia"]},

    # ── clearly_false (6) — 통념 debunk는 Wikipedia 주력, 시사성 음모론은 News 보완,
    #     학술 반박이 명확한 건 ArXiv도 자연스럽게 잡힘.
    {"claim": "인간은 뇌의 10%만 사용한다.",
     "expected_label": "clearly_false", "fact_label": False, "domain": "신경과학",
     "expected_apis": ["wikipedia"]},
    {"claim": "지구는 평평하다.",
     "expected_label": "clearly_false", "fact_label": False, "domain": "과학",
     "expected_apis": ["wikipedia"]},
    {"claim": "1969년 아폴로 11호 달 착륙은 조작이다.",
     "expected_label": "clearly_false", "fact_label": False, "domain": "역사",
     "expected_apis": ["wikipedia"]},
    {"claim": "학술 연구에 따르면 백신은 자폐를 유발한다.",
     "expected_label": "clearly_false", "fact_label": False, "domain": "의학",
     "expected_apis": ["arxiv", "wikipedia"]},
    {"claim": "이버멕틴은 코로나19 치료에 효과적이라는 임상 증거가 충분하다.",
     "expected_label": "clearly_false", "fact_label": False, "domain": "의학/팬데믹",
     "expected_apis": ["arxiv", "news"]},
    {"claim": "최근 보도에 따르면 2024년 미국 대선은 광범위한 부정선거로 결과가 뒤집혔다.",
     "expected_label": "clearly_false", "fact_label": False, "domain": "시사/정치",
     "expected_apis": ["news"]},

    # ── debatable (10) — 핵심 평가 대상. API 분포를 의도적으로 분산.
    #
    # [News 우세 — 최근 1개월 영어 매체에서 실제로 다뤄진 사실에 대한 해석/평가]
    {"claim": "최근 OpenAI의 영리 기업 전환은 AI 안전성을 약화시켰다.",
     "expected_label": "debatable", "fact_label": None, "domain": "AI 산업/거버넌스",
     "expected_apis": ["news"]},
    {"claim": "최근 일론 머스크의 X(트위터) 운영 방침은 표현의 자유를 후퇴시켰다.",
     "expected_label": "debatable", "fact_label": None, "domain": "미디어/플랫폼",
     "expected_apis": ["news"]},
    {"claim": "최근 중국의 희토류 수출 통제 강화는 서방의 반도체 공급망에 심각한 위협이다.",
     "expected_label": "debatable", "fact_label": None, "domain": "국제정치/무역",
     "expected_apis": ["news"]},

    # [ArXiv 우세 — '최근 연구/벤치마크/실증 연구' 같은 학술 신호 포함]
    {"claim": "최근 연구에 따르면 대형 언어 모델은 진정한 의미의 추론을 수행하지 못한다.",
     "expected_label": "debatable", "fact_label": None, "domain": "AI/인지과학",
     "expected_apis": ["arxiv"]},
    {"claim": "강화학습 기반 RLHF는 벤치마크 상에서 LLM 정렬 문제의 근본적 해법으로 입증되었다.",
     "expected_label": "debatable", "fact_label": None, "domain": "AI 안전/학술",
     "expected_apis": ["arxiv"]},
    {"claim": "메타분석에 따르면 청소년의 SNS 사용은 정신건강에 부정적 영향을 준다.",
     "expected_label": "debatable", "fact_label": None, "domain": "보건/심리학 연구",
     "expected_apis": ["arxiv"]},

    # [Wikipedia 우세 — 백과사전적 배경 + 정책 동향 혼합]
    {"claim": "원자력은 탄소중립 달성에 필수적인 에너지원이다.",
     "expected_label": "debatable", "fact_label": None, "domain": "에너지",
     "expected_apis": ["wikipedia", "news"]},
    {"claim": "전기차는 내연기관차보다 환경에 더 이롭다.",
     "expected_label": "debatable", "fact_label": None, "domain": "환경",
     "expected_apis": ["wikipedia", "arxiv"]},
    {"claim": "사형제는 흉악범죄 억제에 효과적이므로 유지되어야 한다.",
     "expected_label": "debatable", "fact_label": None, "domain": "사회/윤리",
     "expected_apis": ["wikipedia", "arxiv"]},

    # [혼합 — 학술 연구 + 최근 동향 둘 다 유의미]
    {"claim": "실증 연구에 따르면 재택근무는 사무실 근무보다 생산성이 높다.",
     "expected_label": "debatable", "fact_label": None, "domain": "노동",
     "expected_apis": ["arxiv", "news"]},

    # [한국 맥락 — 영어 매체 News + 백과사전 배경]
    {"claim": "한국의 저출생 문제는 주거비 부담이 가장 큰 원인이다.",
     "expected_label": "debatable", "fact_label": None, "domain": "한국 사회",
     "expected_apis": ["news", "wikipedia"]},

    # ── subjective_opinion (2) — 사실 라벨 없음. 취향 반박 모드.
    {"claim": "클래식 음악이 재즈보다 더 가치 있는 예술이다.",
     "expected_label": "subjective_opinion", "fact_label": None, "domain": "음악",
     "expected_apis": ["wikipedia"]},
    {"claim": "영화는 책보다 우월한 매체이다.",
     "expected_label": "subjective_opinion", "fact_label": None, "domain": "매체/문화",
     "expected_apis": ["wikipedia"]},
]

# 분류 배분 + 의도된 API 분포 확인
from collections import Counter
_dist = Counter(c["expected_label"] for c in EVAL_CLAIMS)
_api_dist = Counter()
for c in EVAL_CLAIMS:
    for a in c.get("expected_apis", []):
        _api_dist[a] += 1
print(f"Total: {len(EVAL_CLAIMS)}")
for label in ["established_fact", "clearly_false", "debatable", "subjective_opinion"]:
    print(f"  {label}: {_dist[label]}")
print("Intended API coverage (multi-count):")
for api in ["wikipedia", "news", "arxiv"]:
    print(f"  {api}: {_api_dist[api]}")


## 15. 데모 — 단일 호출

`respond()` 한 번 호출해서 동작 확인.


In [16]:
# 데모 1: clearly_false 명제로 1턴
demo_history = []
demo_msg = "백신은 자폐를 유발한다."
result = respond(demo_msg, demo_history)

print("=== Turn 1 ===")
print(f"Classification:    {result['classification']}")
print(f"Evidence strength: {result['evidence_strength']}")
print(f"Evaluator state:   {result['evaluator_state']}")
print(f"Sources ({len(result['sources'])}):")
for s in result["sources"][:3]:
    print(f"  - [{s['source_type']}] {s['title'][:70]}")
print(f"\nReply:\n{result['reply']}")
print(f"\n{llm.summary()}")


=== Turn 1 ===
Classification:    clearly_false
Evidence strength: weak
Evaluator state:   continue
Sources (11):
  - [wikipedia] MMR vaccine and autism
  - [wikipedia] Vaccines and autism
  - [wikipedia] Measles vaccine

Reply:
사용자 주장 핵심 결함은 “백신이 자폐를 유발한다”는 인과관계를 단정하지만, 대규모 역학조사와 건강당국의 검증 결과와 정면으로 배치된다는 점입니다. 제공된 자료에 따르면 자폐와 백신 사이의 관계는 “인과적이든 그렇지 않든” 없다고 광범위하게 조사되어 과학적 합의로 정리되어 있습니다([Wikipedia | Vaccines and autism]). 또한 MMR 백신과 자폐의 연관성 주장은 1990년대 초 처음 제기된 뒤, 1998년 The Lancet의 사기 논문이 큰 계기가 되었고 그 논문은 2010년에 철회되었으며 해당 주장은 거짓으로 판정되었습니다([Wikipedia | MMR vaccine and autism]). 즉 “유발한다”는 결론을 뒷받침할 근거가 아니라, 오히려 반대되는 검증 결과가 제시됩니다([Wikipedia | MMR vaccine and autism], [Wikipedia | Vaccines and autism]). 게다가 백신이 면역계를 “압도”한다는 주장도 합의와 맞지 않는 것으로 정리되어 있습니다([Wikipedia | Vaccines and autism]).  

반박 포인트로, “어떤 백신(예: MMR, MMRV, 특정 성분)”과 “어떤 연구 설계(코호트/메타분석 등)”를 근거로 ‘유발’이라고 말하는지 구체적으로 제시할 수 있나요?

[gpt-5.4-nano] calls=5, in=2987, out=543, cost=$0.0047


## 16. 데모 — 멀티턴 (수동)

In [17]:
# Turn 1 결과를 history에 넣고 Turn 2 진행
demo_history.append({
    "user": demo_msg,
    "reply": result["reply"],
    "sources": result["sources"],
    "classification": result["classification"],
    "evidence_strength": result["evidence_strength"],
    "evaluator_state": result["evaluator_state"],
})

demo_msg2 = "그래도 내 사촌이 백신 맞고 나서 변했다는데, 그건 어떻게 설명할 거야?"
result2 = respond(demo_msg2, demo_history)

print("=== Turn 2 ===")
print(f"Evidence strength: {result2['evidence_strength']}")
print(f"Evaluator state:   {result2['evaluator_state']}")
print(f"\nReply:\n{result2['reply']}")
print(f"\n{llm.summary()}")


=== Turn 2 ===
Evidence strength: weak
Evaluator state:   continue

Reply:
핵심 결함은 “사촌이 백신 후 변했다”는 **개별 사례**를 “백신이 자폐를 유발한다”는 **인과 결론**으로 바로 연결한다는 점이야. 백신 접종 시기와 증상(또는 행동 변화)이 겹칠 수는 있지만, 그게 곧바로 인과를 증명하진 않아. 실제로 자폐와 백신의 관계는 대규모 조사에서 **인과적 관련이 없다는 과학적 합의**로 정리돼 있어([Wikipedia | Vaccines and autism]). 또한 MMR-자폐 주장처럼, 과거에 특정 사건/논문이 퍼지며 인과처럼 보이게 만든 사례가 있었고, 그 대표적 예는 1998년 사기 논문이었으며 2010년에 철회됐어([Wikipedia | MMR vaccine and autism]). “변했다”는 관찰은 충분히 이해되지만, 과학적으로는 **우연한 동시성, 기존 발달 과정의 타이밍, 다른 요인**을 배제하지 못하면 원인으로 단정할 수 없어.  

반박 포인트로 하나만 물어볼게: 사촌의 “변화”가 정확히 어떤 진단(예: 자폐 진단)인지, 그리고 그 변화를 백신 접종 전에도 보였는지(발달 기록/의사 소견) 확인할 수 있어?

[gpt-5.4-nano] calls=9, in=5237, out=984, cost=$0.0084


## 17. 데모 — User Simulator로 자동 멀티턴

`run_evaluation`을 명제 1개에 대해 돌려본다. 실제 평가는
명제 20개로 별도 실행.


In [ ]:
# 시뮬레이터 데모 — 3개 외부 API + 3개 카테고리 시연
#   1) clearly_false + Wikipedia: 통념 debunk
#   2) clearly_false + News:   최근 보도 사실 확인
#   3) debatable    + ArXiv:     학술 논쟁

demo_claim_set = [
    {"claim": "인간은 뇌의 10%만 사용한다.",
     "expected_label": "clearly_false",
     "expected_apis": ["wikipedia"]},
    {"claim": "Nvidia의 2026년 1분기 데이터센터 부문 매출은 주요 고객사들의 AI 인프라 투자 축소로 인해 전년 동기 대비 20% 감소했습니다.",
     "expected_label": "clearly_false",
     "expected_apis": ["news"]},
    {"claim": "최근 연구에 따르면 대형 언어 모델은 진정한 의미의 추론을 수행하지 못한다.",
     "expected_label": "debatable",
     "expected_apis": ["arxiv"]},
]

eval_results = run_evaluation(demo_claim_set, persona="believer",
                              max_turns=MAX_TURNS)

print("\n\n=== 시뮬레이션 결과 ===")
for r in eval_results:
    print(f"\n명제: {r['claim_item']['claim']}")
    print(f"  예상 API: {r['claim_item'].get('expected_apis', [])}")
    api_used = {}
    for h in r['history']:
        for s in h.get('sources', []):
            t = s.get('source_type', '?')
            api_used[t] = api_used.get(t, 0) + 1
    print(f"  실제 호출 API 분포: {api_used}")
    for i, h in enumerate(r['history'], 1):
        print(f"  Turn {i}")
        print(f"    USER:    {h['user'][:100]}")
        print(f"    BOT:     {h['reply'][:100]}")
        # 이 턴에서 검색된 문서 제목 출력 (어떤 내용을 가져왔는지 확인)
        if h.get('sources'):
            print(f"    SOURCES ({len(h['sources'])}):")
            for s in h['sources'][:5]:  # 너무 많으면 5개까지만
                src_type = s.get('source_type', '?')
                title = s.get('title', '(no title)')[:80]
                print(f"      [{src_type}] {title}")
        else:
            print(f"    SOURCES: (none)")

print(f"\n{llm.summary()}")
print(f"{simulator_llm.summary()}")


## 18. 마무리 — 비용/호출 요약

`respond()` 호출 횟수와 누적 비용 확인. 명세 §4.3 가드레일.


In [19]:
print("=== Chatbot LLM ===")
print(f"Calls:         {llm.call_count}")
print(f"Input tokens:  {llm.input_tokens:,}")
print(f"Output tokens: {llm.output_tokens:,}")
print(f"Cost:          ${llm.cost_usd():.4f}")

print("\n=== Simulator LLM ===")
print(f"Calls:         {simulator_llm.call_count}")
print(f"Input tokens:  {simulator_llm.input_tokens:,}")
print(f"Output tokens: {simulator_llm.output_tokens:,}")
print(f"Cost:          ${simulator_llm.cost_usd():.4f}")

total = llm.cost_usd() + simulator_llm.cost_usd()
print(f"\n총 비용:       ${total:.4f}")
print(f"예산 한도:     $20.00")
print(f"경고 임계치:   ${COST_WARN_THRESHOLD}")


=== Chatbot LLM ===
Calls:         57
Input tokens:  40,981
Output tokens: 7,241
Cost:          $0.0633

=== Simulator LLM ===
Calls:         9
Input tokens:  6,598
Output tokens: 1,405
Cost:          $0.0113

총 비용:       $0.0746
예산 한도:     $20.00
경고 임계치:   $9999.0


# evaluation

### Dry-run 토글

`DRY_RUN_N`을 정수로 설정하면 `EVAL_CLAIMS`를 그 개수만큼만 잘라 평가 본 실행에 사용.
오류 탐지·시간 절약용. `None`이면 명세 §5.1에 따라 20개 모두 실행.

In [ ]:
# Dry-run subset 토글: None이면 전체, 정수면 처음 N개만 사용.
# 오류 탐지나 빠른 확인 시 2~5로 설정.
DRY_RUN_N = None  # ← 예: 2 (dry-run), None (전체 20개)

if DRY_RUN_N is not None:
    EVAL_CLAIMS = EVAL_CLAIMS[:DRY_RUN_N]
    print(f"[DRY RUN] EVAL_CLAIMS truncated to {len(EVAL_CLAIMS)} claims:")
    for c in EVAL_CLAIMS:
        print(f"  - [{c['expected_label']}] {c['claim']}")
else:
    print(f"[FULL RUN] Using all {len(EVAL_CLAIMS)} EVAL_CLAIMS")


In [ ]:
# ============================================================
# §19. 전체 평가 실행 — EVAL_CLAIMS × PERSONAS_TO_RUN 멀티턴 대화 + 로그 저장
# ============================================================
# 각 (페르소나 × 명제) 조합마다 run_evaluation을 1회 호출, 결과를 페르소나별 디렉터리에 저장:
#   eval_outputs/{persona}/log_NN.txt     — 사람이 읽을 멀티턴 대화 로그
#   eval_outputs/{persona}/result_NN.txt  — 메타 정보 (분류·근거강도·sources·API 분포 등)
#
# 시작 옵션:
#   PERSONAS_TO_RUN = ["believer"]                                  # 단일 페르소나 (빠른 테스트)
#   PERSONAS_TO_RUN = ["believer","moderate","stubborn","evidence_focused"]   # 명세 §9.2 (4페르소나)
#
# 비용 추산 (gpt-5.4-nano 기준):
#   - 1페르소나: ~$0.5 (eval) + ~$0.06 (baseline) + ~$0.2 (judge) = ~$0.8
#   - 4페르소나: ~$3.0 total
# 소요 시간:
#   - 1페르소나: ~40분
#   - 4페르소나: ~2.5~3시간

import os
import json
from collections import Counter
from datetime import datetime

# 명제 개수 확인 (명세: 20개. EVAL_CLAIMS가 21개면 마지막 1개 제거)
_claim_check = len(EVAL_CLAIMS)
print(f"Total claims in EVAL_CLAIMS: {_claim_check}")
if _claim_check > 20:
    print(f"⚠ 명세에서는 20개이나 현재 {_claim_check}개. 마지막 명제 제거하겠습니다.")
    EVAL_CLAIMS = EVAL_CLAIMS[:20]
    print(f"Adjusted to: {len(EVAL_CLAIMS)} claims")

# ── 평가 대상 페르소나 (수정해서 실행 범위 조정) ────────────────────────
PERSONAS_TO_RUN = ["believer", "moderate", "stubborn", "evidence_focused"]
# 빠르게 1페르소나만: PERSONAS_TO_RUN = ["believer"]

persona_to_results = {}  # {"believer": [...], "moderate": [...], ...} — 셀 51이 사용

for PERSONA in PERSONAS_TO_RUN:
    OUT_DIR = f"eval_outputs/{PERSONA}"
    os.makedirs(OUT_DIR, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"=== Evaluation start | persona={PERSONA} | claims={len(EVAL_CLAIMS)} ===")
    print(f"Output dir: {OUT_DIR}/")
    print(f"{'='*70}\n")

    # 페르소나별 비용 스냅샷
    _cost_start_bot = llm.cost_usd()
    _cost_start_sim = simulator_llm.cost_usd()
    _calls_start_bot = llm.call_count
    _calls_start_sim = simulator_llm.call_count

    all_results = []

    for idx, claim_item in enumerate(EVAL_CLAIMS, start=1):
        tag = f"{idx:02d}"
        claim = claim_item["claim"]
        expected_apis = claim_item.get("expected_apis", [])

        _c0_bot = llm.cost_usd()
        _c0_sim = simulator_llm.cost_usd()

        try:
            sub_results = run_evaluation([claim_item], persona=PERSONA, max_turns=MAX_TURNS)
            history = sub_results[0]["history"]
            error = None
        except Exception as e:
            history = []
            error = f"{type(e).__name__}: {e}"
            print(f"  [ERROR] {claim[:50]} — {error}")

        _c1_bot = llm.cost_usd()
        _c1_sim = simulator_llm.cost_usd()
        item_cost = (_c1_bot - _c0_bot) + (_c1_sim - _c0_sim)

        api_used = Counter()
        for h in history:
            for s in h.get("sources", []):
                api_used[s.get("source_type", "?")] += 1
        api_match = bool(set(expected_apis) & set(api_used.keys())) if expected_apis else None

        # ─── log_NN.txt ────────────────────────────────────────────────
        log_path = os.path.join(OUT_DIR, f"log_{tag}.txt")
        with open(log_path, "w", encoding="utf-8") as f:
            f.write(f"=== Claim {tag} ===\n")
            f.write(f"Claim:          {claim}\n")
            f.write(f"Expected label: {claim_item['expected_label']}\n")
            f.write(f"Fact label:     {claim_item['fact_label']}\n")
            f.write(f"Domain:         {claim_item['domain']}\n")
            f.write(f"Expected APIs:  {expected_apis}\n")
            f.write(f"Persona:        {PERSONA}\n")
            f.write(f"Timestamp:      {datetime.now().isoformat(timespec='seconds')}\n")
            f.write("=" * 70 + "\n\n")
            if error:
                f.write(f"[ERROR] {error}\n")
            for t, h in enumerate(history, start=1):
                f.write(f"--- Turn {t} ---\n")
                f.write(f"USER: {h['user']}\n\n")
                f.write(f"BOT:  {h['reply']}\n\n")

        # ─── result_NN.txt ─────────────────────────────────────────────
        result_path = os.path.join(OUT_DIR, f"result_{tag}.txt")
        with open(result_path, "w", encoding="utf-8") as f:
            f.write(f"=== Claim {tag} — Result Metadata ===\n")
            f.write(f"Claim:           {claim}\n")
            f.write(f"Expected label:  {claim_item['expected_label']}\n")
            f.write(f"Fact label:      {claim_item['fact_label']}\n")
            f.write(f"Domain:          {claim_item['domain']}\n")
            f.write(f"Persona:         {PERSONA}\n")
            f.write(f"Turns completed: {len(history)} / {MAX_TURNS}\n")
            f.write(f"Cost (this item): ${item_cost:.4f}\n")
            if error:
                f.write(f"Error:           {error}\n")
            f.write("\n")
            f.write(f"Expected APIs:   {expected_apis}\n")
            f.write(f"Actual APIs:     {dict(api_used)}\n")
            f.write(f"API match:       {api_match}\n\n")
            if history:
                f.write(f"Classification (turn 1): {history[0]['classification']}\n")
                f.write(f"Final evaluator state:   {history[-1]['evaluator_state']}\n\n")
            for t, h in enumerate(history, start=1):
                f.write(f"--- Turn {t} ---\n")
                f.write(f"  classification:    {h['classification']}\n")
                f.write(f"  evidence_strength: {h['evidence_strength']}\n")
                f.write(f"  evaluator_state:   {h['evaluator_state']}\n")
                f.write(f"  sources ({len(h['sources'])}):\n")
                for s in h["sources"]:
                    f.write(f"    - [{s.get('source_type','?')}] {s.get('title','')[:80]}\n")
                    if s.get("url"):
                        f.write(f"      {s['url']}\n")
                f.write("\n")

        all_results.append({
            "tag": tag,
            "claim_item": claim_item,
            "history": history,
            "error": error,
            "cost": item_cost,
            "api_used": dict(api_used),
            "api_match": api_match,
        })
        match_mark = "✓" if api_match else ("✗" if api_match is False else "—")
        print(f"  [{tag}/{len(EVAL_CLAIMS)}] {'✓' if not error else '✗'} "
              f"{claim[:50]:<50} turns={len(history)} "
              f"api_match={match_mark} cost=${item_cost:.4f}")

    persona_to_results[PERSONA] = all_results

    # ─── 페르소나별 요약 ───────────────────────────────────────────
    print(f"\n=== {PERSONA} done ===")
    print(f"Chatbot calls:    {llm.call_count - _calls_start_bot} "
          f"(+${llm.cost_usd() - _cost_start_bot:.4f})")
    print(f"Simulator calls:  {simulator_llm.call_count - _calls_start_sim} "
          f"(+${simulator_llm.cost_usd() - _cost_start_sim:.4f})")

    _eval_match = [r["api_match"] for r in all_results if r["api_match"] is not None]
    if _eval_match:
        _hits = sum(_eval_match)
        print(f"Router API match: {_hits}/{len(_eval_match)} "
              f"({_hits/len(_eval_match)*100:.0f}%)")

    _total_apis = Counter()
    for r in all_results:
        for k, v in r["api_used"].items():
            _total_apis[k] += v
    print(f"Total API calls: {dict(_total_apis)}")
    print(f"Files saved to {OUT_DIR}/: log_01~{len(EVAL_CLAIMS):02d}.txt, result_01~{len(EVAL_CLAIMS):02d}.txt")

# 마지막 페르소나 결과를 all_results에 alias (단일 페르소나 모드에서 셀 51 호환)
all_results = persona_to_results[PERSONAS_TO_RUN[-1]]
print(f"\n{'='*70}\n### ALL PERSONAS DONE: {PERSONAS_TO_RUN} ###\n{'='*70}")


## §20. 평가 지표 — Judge 프롬프트 (명세 §9.2)

5종 지표 산출에 쓰일 LLM judge 프롬프트. 모두 strict JSON 출력 강제.
- `judge_persuasion` : believer 사용자의 입장 변화 (0 / 0.5 / 1.0)
- `judge_soundness` : 챗봇 응답의 논리적 정합성 (1~5)
- `judge_persuasiveness` : 챗봇 응답의 설득력 (1~5)
- `judge_winrate` : 우리 챗봇 vs grounding 없는 일반 ChatGPT 비교


In [ ]:
JUDGE_PROMPTS = {

"judge_persuasion": """당신은 토론 평가자이다. 사용자(believer 페르소나)가 처음 던진 주장과 마지막 발화를 비교하여
사용자의 입장이 얼마나 흔들렸는지 평가한다.

평가 기준:
- 0   : 입장 불변. 여전히 원 주장을 동일한 강도로 옹호.
- 0.5 : 부분적 의문 표출, 일부 반례 인정, 톤이 누그러짐.
- 1.0 : 완전 철회 또는 챗봇 입장 수용.

반드시 다음 JSON 형식으로만 응답:
{"score": <0|0.5|1.0>, "rationale": "<1~2문장 근거>"}
""",

"judge_soundness": """당신은 토론 평가자이다. 챗봇 응답의 논리적 정합성을 평가한다.

평가 기준 (1~5):
- 5 : 명료한 논증, 제시된 근거가 결론을 직접 지지, 오류 없음.
- 4 : 거의 정합. 사소한 비약 1개 이하.
- 3 : 보통. 핵심 주장은 일관되나 일부 단순화·과장.
- 2 : 약함. 결론과 근거가 잘 안 맞거나 논리적 비약 다수.
- 1 : 모순·오류·환각이 두드러짐.

context_snippets에 없는 사실을 단정적으로 단언하면 감점.

반드시 다음 JSON 형식으로만 응답:
{"score": <1|2|3|4|5>, "rationale": "<1~2문장 근거>"}
""",

"judge_persuasiveness": """당신은 토론 평가자이다. 챗봇 응답이 얼마나 설득력 있는지 평가한다.
대상 청중은 원 주장을 굳게 믿는 believer 사용자이다.

평가 기준 (1~5):
- 5 : 매우 설득력. believer가 흔들릴 만한 구체 근거·강한 수사·명확한 호소.
- 4 : 설득력 있음. 근거 인용 + 명료한 반론.
- 3 : 보통. 일반론적 반박. 새로운 통찰은 적음.
- 2 : 약함. 모호하거나 일방적 단정.
- 1 : 설득 의지 없음 또는 자기 모순.

반드시 다음 JSON 형식으로만 응답:
{"score": <1|2|3|4|5>, "rationale": "<1~2문장 근거>"}
""",

"judge_winrate": """당신은 공정한 토론 심판이다. 동일한 사용자 주장에 대한 두 응답(A, B)을 비교하여
어느 쪽이 더 우수한지 판정한다.

평가 기준:
- 사실 정확성, 근거의 구체성, 논리적 정합성, 설득력을 종합.
- 어느 쪽이 더 많이 구체 출처·수치를 인용하는지.
- 어느 쪽이 더 균형 잡힌 반박을 제공하는지.

응답자 A, B 중 하나의 정체를 추론하려 들지 말 것. 오직 내용만으로 판정.

반드시 다음 JSON 형식으로만 응답:
{"winner": "A"|"B"|"tie", "rationale": "<1~2문장 근거>"}
""",
}

print(f"Judge prompts loaded: {len(JUDGE_PROMPTS)} entries")


## §21. Baseline — Grounding 없는 일반 ChatGPT 응답

각 명제에 대해 외부 검색·인용 없이 단발 응답을 생성한다.
명세 §5.4의 Debate Win-rate 비교 대상.


In [ ]:
# Baseline LLM 클라이언트 (비용 분리 추적)
baseline_llm = LLMClient(model=MODEL)

BASELINE_SYSTEM = """당신은 ChatGPT입니다. 사용자의 주장에 대해 4~6문장으로 응답하세요.
외부 검색·인용·URL 출처는 사용하지 마십시오. 일반 지식과 상식만으로 답합니다.
사용자가 사용한 언어 그대로 응답합니다."""

def baseline_response(claim: str) -> str:
    """Grounding 없는 단발 ChatGPT 응답 생성."""
    return baseline_llm.call(BASELINE_SYSTEM, claim,
                             temperature=0.5, max_tokens=400)


# 20개 명제 모두에 대해 baseline 응답 생성 (캐시)
baseline_replies: dict[str, str] = {}
print(f"=== Baseline 응답 생성: {len(EVAL_CLAIMS)}개 명제 ===")
for idx, claim_item in enumerate(EVAL_CLAIMS, start=1):
    tag = f"{idx:02d}"
    try:
        baseline_replies[tag] = baseline_response(claim_item["claim"])
        print(f"  [{tag}] OK ({len(baseline_replies[tag])} chars)")
    except Exception as e:
        baseline_replies[tag] = ""
        print(f"  [{tag}] FAIL: {type(e).__name__}: {e}")

print(f"\nBaseline LLM 비용: {baseline_llm.summary()}")


## §22. 5종 지표 계산 함수 (명세 §9.2)

- `metric_classification_accuracy` : LLM 호출 0회. 라벨 일치 비율.
- `metric_persuasion` : 명제당 judge 1회.
- `metric_soundness` : 명제당 judge 호출 = 턴 수.
- `metric_persuasiveness` : 명제당 judge 호출 = 턴 수.
- `metric_winrate` : 명제당 judge 1회. A/B 위치 무작위화.


In [ ]:
import random
from collections import Counter

# Judge LLM 클라이언트 (챗봇·시뮬레이터와 비용 분리)
judge_llm = LLMClient(model=JUDGE_MODEL)


# ─── 지표 1: Classification Accuracy (LLM 호출 0회) ────────────────
def metric_classification_accuracy(all_results: list[dict]) -> dict:
    """expected_label vs 실제 분류(첫 턴) 비교. 모든 20개 명제 평가."""
    rows = []
    correct = 0
    counted = 0
    confusion = Counter()  # (expected, actual) -> count
    for r in all_results:
        tag = r["tag"]
        expected = r["claim_item"]["expected_label"]
        if not r["history"]:
            rows.append({"tag": tag, "expected": expected,
                         "actual": None, "correct": None})
            continue
        actual = r["history"][0]["classification"]
        is_correct = (actual == expected)
        counted += 1
        if is_correct:
            correct += 1
        confusion[(expected, actual)] += 1
        rows.append({"tag": tag, "expected": expected,
                     "actual": actual, "correct": is_correct})
    accuracy = (correct / counted) if counted else 0.0
    return {"accuracy": accuracy, "correct": correct, "total": counted,
            "rows": rows, "confusion": dict(confusion)}


# ─── 지표 2: Persuasion Score (명제당 judge 1회) ───────────────────
def metric_persuasion(result: dict) -> dict | None:
    """첫 발화 vs 마지막 발화로 입장 변화 평가. 1턴짜리는 None."""
    history = result["history"]
    if len(history) <= 1:
        return None  # established_fact 즉시 종료 — believer가 반응할 기회 없음
    claim = result["claim_item"]["claim"]
    first_msg = history[0]["user"]
    last_msg = history[-1]["user"]
    user_prompt = (f"원 주장(claim): {claim}\n\n"
                   f"사용자 최초 발화: {first_msg}\n\n"
                   f"사용자 마지막 발화: {last_msg}\n\n"
                   f"대화 턴 수: {len(history)}")
    raw = judge_llm.call_json(JUDGE_PROMPTS["judge_persuasion"], user_prompt)
    score = raw.get("score")
    if score not in [0, 0.5, 1.0, 0.0, 1]:
        print(f"  [WARN persuasion] invalid score {score!r}, fallback 0")
        score = 0
        raw = {"score": 0, "rationale": "[FALLBACK] invalid score"}
    return {"score": float(score), "raw": raw}


# ─── 지표 3: Logical Soundness (명제당 judge 호출 = 턴 수) ─────────
def _format_snippets_for_judge(sources: list[dict], k: int = 5) -> str:
    if not sources:
        return "(검색 결과 없음)"
    lines = []
    for i, s in enumerate(sources[:k], 1):
        lines.append(f"[{i}] {s.get('source_type','?')} | "
                     f"{s.get('title','')[:80]}\n"
                     f"    {(s.get('snippet','') or '')[:300]}")
    return "\n".join(lines)


def metric_soundness(result: dict) -> dict:
    """턴별 1~5 점수의 평균. 컨텍스트(검색 snippets) 포함하여 평가."""
    history = result["history"]
    if not history:
        return {"score": None, "per_turn": []}
    claim = result["claim_item"]["claim"]
    per_turn = []
    for t, h in enumerate(history, start=1):
        ctx = _format_snippets_for_judge(h.get("sources", []))
        user_prompt = (f"원 주장: {claim}\n\n"
                       f"턴 {t} 챗봇 응답:\n{h['reply']}\n\n"
                       f"검색된 context_snippets:\n{ctx}")
        raw = judge_llm.call_json(JUDGE_PROMPTS["judge_soundness"], user_prompt)
        s = raw.get("score")
        if not isinstance(s, (int, float)) or not (1 <= s <= 5):
            print(f"  [WARN soundness T{t}] invalid {s!r}, fallback 3")
            s = 3
            raw = {"score": 3, "rationale": "[FALLBACK]"}
        per_turn.append({"turn": t, "score": float(s), "raw": raw})
    avg = sum(x["score"] for x in per_turn) / len(per_turn)
    return {"score": avg, "per_turn": per_turn}


# ─── 지표 4: Persuasiveness (명제당 judge 호출 = 턴 수) ────────────
def metric_persuasiveness(result: dict) -> dict:
    """턴별 1~5 점수의 평균. believer 청중 가정."""
    history = result["history"]
    if not history:
        return {"score": None, "per_turn": []}
    claim = result["claim_item"]["claim"]
    per_turn = []
    for t, h in enumerate(history, start=1):
        user_prompt = (f"원 주장(사용자 입장): {claim}\n"
                       f"청중: 원 주장을 굳게 믿는 believer 사용자\n\n"
                       f"턴 {t} 챗봇 응답:\n{h['reply']}")
        raw = judge_llm.call_json(JUDGE_PROMPTS["judge_persuasiveness"], user_prompt)
        s = raw.get("score")
        if not isinstance(s, (int, float)) or not (1 <= s <= 5):
            print(f"  [WARN persuasiveness T{t}] invalid {s!r}, fallback 3")
            s = 3
            raw = {"score": 3, "rationale": "[FALLBACK]"}
        per_turn.append({"turn": t, "score": float(s), "raw": raw})
    avg = sum(x["score"] for x in per_turn) / len(per_turn)
    return {"score": avg, "per_turn": per_turn}


# ─── 지표 5: Debate Win-rate (명제당 judge 1회, A/B 위치 무작위) ───
def metric_winrate(result: dict, baseline_reply: str,
                   rng: random.Random | None = None) -> dict:
    """우리 챗봇 첫 턴 응답 vs baseline 단발 응답. A/B 위치 무작위화."""
    history = result["history"]
    if not history or not baseline_reply:
        return {"winner": None, "raw": None}
    rng = rng or random
    claim = result["claim_item"]["claim"]
    our_reply = history[0]["reply"]
    # 위치 무작위화
    if rng.random() < 0.5:
        a_label, b_label = "ours", "baseline"
        a_text, b_text = our_reply, baseline_reply
    else:
        a_label, b_label = "baseline", "ours"
        a_text, b_text = baseline_reply, our_reply
    user_prompt = (f"사용자 주장: {claim}\n\n"
                   f"=== 응답 A ===\n{a_text}\n\n"
                   f"=== 응답 B ===\n{b_text}")
    raw = judge_llm.call_json(JUDGE_PROMPTS["judge_winrate"], user_prompt)
    winner_ab = raw.get("winner")
    if winner_ab not in {"A", "B", "tie"}:
        print(f"  [WARN winrate] invalid {winner_ab!r}, fallback tie")
        winner_ab = "tie"
        raw = {"winner": "tie", "rationale": "[FALLBACK]"}
    if winner_ab == "tie":
        winner = "tie"
    elif winner_ab == "A":
        winner = a_label
    else:
        winner = b_label
    return {"winner": winner, "ab_assignment": {"A": a_label, "B": b_label},
            "raw": raw}


print("Metric functions defined: classification / persuasion / soundness / "
      "persuasiveness / winrate")


## §23. 평가 실행 및 결과 표 생성

`all_results`(셀 41 산출물)와 `baseline_replies`(셀 B 산출물)를 받아
5종 지표를 모두 계산하고 `eval_outputs/metrics_summary.txt`에 표로 저장.
명제별 judge raw output(rationale 포함)은 `eval_outputs/metrics_detail_NN.txt`에 별도 저장.


In [ ]:
# ============================================================
# §23. 평가 지표 계산 — 페르소나별로 5종 지표 산출 및 저장
# ============================================================
# PERSONAS_TO_RUN(셀 43에서 정의)을 따라 모든 페르소나의 metrics_summary.txt 생성.
# persona_to_results dict에서 각 페르소나의 all_results를 가져와 사용.

import os
import statistics

_rng = random.Random(42)  # win-rate A/B 무작위화 (재현성)

for PERSONA in PERSONAS_TO_RUN:
    OUT_DIR = f"eval_outputs/{PERSONA}"
    os.makedirs(OUT_DIR, exist_ok=True)
    all_results = persona_to_results[PERSONA]  # 이 페르소나의 eval 결과

    print(f"\n{'='*70}")
    print(f"### Computing metrics for persona: {PERSONA} ###")
    print(f"{'='*70}")

    # 페르소나별 judge 비용 스냅샷
    _judge_calls_start = judge_llm.call_count
    _judge_cost_start = judge_llm.cost_usd()

    # ─── 1. Classification Accuracy (LLM 호출 0회) ────────────────────
    print(f"\n=== 1/5 Classification Accuracy ===")
    cls_result = metric_classification_accuracy(all_results)
    print(f"  Accuracy: {cls_result['correct']}/{cls_result['total']} "
          f"({cls_result['accuracy']*100:.1f}%)")

    # ─── 2~5. LLM judge 지표들 — 명제별로 계산 ────────────────────────
    metric_rows = []
    detail_records = []

    for idx, r in enumerate(all_results, start=1):
        tag = r["tag"]
        claim = r["claim_item"]["claim"]
        print(f"\n[{tag}/{len(all_results)}] {claim[:50]}")
        err = r.get("error")
        if err or not r["history"]:
            print(f"  ERROR or empty history — skip LLM metrics")
            metric_rows.append({
                "tag": tag, "claim": claim,
                "cls_correct": None, "persuasion": None,
                "soundness_avg": None, "persuasiveness_avg": None,
                "winrate_outcome": None, "error": err or "empty history",
            })
            continue

        cls_row = next((x for x in cls_result["rows"] if x["tag"] == tag), None)
        cls_correct = cls_row["correct"] if cls_row else None
        print(f"  cls: expected={cls_row['expected']} actual={cls_row['actual']} "
              f"→ {'✓' if cls_correct else '✗'}")

        pers = metric_persuasion(r)
        pers_score = pers["score"] if pers else None
        print(f"  persuasion: {pers_score}")

        snd = metric_soundness(r)
        snd_avg = snd["score"]
        print(f"  soundness avg: {snd_avg:.2f}" if snd_avg else "  soundness: N/A")

        pwr = metric_persuasiveness(r)
        pwr_avg = pwr["score"]
        print(f"  persuasiveness avg: {pwr_avg:.2f}" if pwr_avg else "  persuasiveness: N/A")

        baseline_reply = baseline_replies.get(tag, "")
        wr = metric_winrate(r, baseline_reply, rng=_rng)
        print(f"  winrate: {wr['winner']}")

        metric_rows.append({
            "tag": tag, "claim": claim,
            "cls_correct": cls_correct,
            "persuasion": pers_score,
            "soundness_avg": snd_avg,
            "persuasiveness_avg": pwr_avg,
            "winrate_outcome": wr["winner"],
            "error": None,
        })
        detail_records.append({
            "tag": tag, "claim": claim,
            "classification": cls_row,
            "persuasion": pers,
            "soundness": snd,
            "persuasiveness": pwr,
            "winrate": wr,
            "baseline_reply": baseline_reply,
            "our_first_reply": r["history"][0]["reply"],
            "user_first": r["history"][0]["user"],
            "user_last": r["history"][-1]["user"],
        })

    # ─── Aggregate ────────────────────────────────────────────────────
    valid_pers = [x["persuasion"] for x in metric_rows if x["persuasion"] is not None]
    valid_snd = [x["soundness_avg"] for x in metric_rows if x["soundness_avg"] is not None]
    valid_pwr = [x["persuasiveness_avg"] for x in metric_rows if x["persuasiveness_avg"] is not None]
    wr_outcomes = [x["winrate_outcome"] for x in metric_rows if x["winrate_outcome"] is not None]
    wr_wins = sum(1 for x in wr_outcomes if x == "ours")
    wr_losses = sum(1 for x in wr_outcomes if x == "baseline")
    wr_ties = sum(1 for x in wr_outcomes if x == "tie")
    wr_total = len(wr_outcomes)
    wr_rate = (wr_wins + 0.5 * wr_ties) / wr_total if wr_total else 0.0

    aggregate = {
        "classification_accuracy": cls_result["accuracy"],
        "persuasion_mean": statistics.mean(valid_pers) if valid_pers else None,
        "soundness_mean": statistics.mean(valid_snd) if valid_snd else None,
        "persuasiveness_mean": statistics.mean(valid_pwr) if valid_pwr else None,
        "winrate": wr_rate,
        "winrate_breakdown": {"ours": wr_wins, "baseline": wr_losses, "tie": wr_ties, "total": wr_total},
    }

    # ─── 표 저장 ───────────────────────────────────────────────────────
    summary_path = os.path.join(OUT_DIR, "metrics_summary.txt")
    header = f"{'tag':<4} {'cls':<4} {'pers':<6} {'snd':<6} {'pwr':<6} {'winrate':<10} claim"
    sep = "-" * 100

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write("=== HW2 평가 지표 요약 (명세 §9.2 5종) ===\n")
        f.write(f"Persona: {PERSONA} | Claims: {len(all_results)} | "
                f"Max turns: {MAX_TURNS}\n\n")
        f.write(header + "\n")
        f.write(sep + "\n")
        for row in metric_rows:
            cls_s = "—" if row["cls_correct"] is None else ("✓" if row["cls_correct"] else "✗")
            pers_s = "—" if row["persuasion"] is None else f"{row['persuasion']}"
            snd_s = "—" if row["soundness_avg"] is None else f"{row['soundness_avg']:.2f}"
            pwr_s = "—" if row["persuasiveness_avg"] is None else f"{row['persuasiveness_avg']:.2f}"
            wr_s = row["winrate_outcome"] or "—"
            f.write(f"{row['tag']:<4} {cls_s:<4} {pers_s:<6} {snd_s:<6} "
                    f"{pwr_s:<6} {wr_s:<10} {row['claim'][:50]}\n")
        f.write(sep + "\n")
        f.write(f"\n=== AGGREGATE ===\n")
        f.write(f"Classification Accuracy : "
                f"{aggregate['classification_accuracy']*100:.1f}% "
                f"({cls_result['correct']}/{cls_result['total']})\n")
        f.write(f"Persuasion (mean)       : "
                f"{aggregate['persuasion_mean']:.3f}\n" if aggregate['persuasion_mean'] is not None
                else "Persuasion (mean)       : N/A\n")
        f.write(f"Logical Soundness (mean): "
                f"{aggregate['soundness_mean']:.2f} / 5\n" if aggregate['soundness_mean'] is not None
                else "Logical Soundness (mean): N/A\n")
        f.write(f"Persuasiveness (mean)   : "
                f"{aggregate['persuasiveness_mean']:.2f} / 5\n" if aggregate['persuasiveness_mean'] is not None
                else "Persuasiveness (mean)   : N/A\n")
        f.write(f"Debate Win-rate         : {aggregate['winrate']*100:.1f}% "
                f"(W{wr_wins}/L{wr_losses}/T{wr_ties} of {wr_total})\n")
        f.write(f"\n=== Classification Confusion ===\n")
        for (exp, act), n in sorted(cls_result["confusion"].items()):
            f.write(f"  {exp:<22} → {act:<22} : {n}\n")
        f.write(f"\n=== Cost (이 페르소나 metrics 실행 분) ===\n")
        f.write(f"Judge LLM: +{judge_llm.call_count - _judge_calls_start} calls, "
                f"+${judge_llm.cost_usd() - _judge_cost_start:.4f}\n")
        f.write(f"Baseline LLM cumulative: {baseline_llm.summary()}\n")

    print(f"\nSummary saved → {summary_path}")

    # ─── 명제별 raw detail 저장 (rationale 포함) ──────────────────────
    for d in detail_records:
        path = os.path.join(OUT_DIR, f"metrics_detail_{d['tag']}.txt")
        with open(path, "w", encoding="utf-8") as f:
            f.write(f"=== Claim {d['tag']} — Metrics Detail ({PERSONA}) ===\n")
            f.write(f"Claim: {d['claim']}\n\n")
            f.write(f"-- Classification --\n")
            f.write(f"  expected: {d['classification']['expected']}\n")
            f.write(f"  actual  : {d['classification']['actual']}\n")
            f.write(f"  correct : {d['classification']['correct']}\n\n")
            f.write(f"-- Persuasion --\n")
            if d["persuasion"]:
                f.write(f"  score: {d['persuasion']['score']}\n")
                f.write(f"  rationale: {d['persuasion']['raw'].get('rationale','')}\n")
                f.write(f"  user_first: {d['user_first']}\n")
                f.write(f"  user_last : {d['user_last']}\n\n")
            else:
                f.write(f"  (1-turn history, skipped)\n\n")
            f.write(f"-- Soundness (avg = {d['soundness']['score']:.2f}) --\n")
            for pt in d["soundness"]["per_turn"]:
                f.write(f"  T{pt['turn']}: {pt['score']:.1f} — "
                        f"{pt['raw'].get('rationale','')}\n")
            f.write(f"\n-- Persuasiveness (avg = {d['persuasiveness']['score']:.2f}) --\n")
            for pt in d["persuasiveness"]["per_turn"]:
                f.write(f"  T{pt['turn']}: {pt['score']:.1f} — "
                        f"{pt['raw'].get('rationale','')}\n")
            f.write(f"\n-- Win-rate --\n")
            f.write(f"  winner: {d['winrate']['winner']}\n")
            f.write(f"  AB assignment: {d['winrate']['ab_assignment']}\n")
            f.write(f"  rationale: {d['winrate']['raw'].get('rationale','')}\n")
            f.write(f"\n-- Reply texts --\n")
            f.write(f"OUR (first turn):\n{d['our_first_reply']}\n\n")
            f.write(f"BASELINE:\n{d['baseline_reply']}\n")

    print(f"Per-claim detail saved: metrics_detail_01.txt ~ "
          f"metrics_detail_{len(all_results):02d}.txt")

# ─── 모든 페르소나 끝난 뒤 cross-persona 요약 출력 ────────────────────
print(f"\n{'='*70}\n### CROSS-PERSONA SUMMARY ###\n{'='*70}")
for PERSONA in PERSONAS_TO_RUN:
    p_summary = os.path.join("eval_outputs", PERSONA, "metrics_summary.txt")
    if os.path.exists(p_summary):
        with open(p_summary, "r", encoding="utf-8") as f:
            content = f.read()
        # AGGREGATE 섹션만 추출해 출력
        if "=== AGGREGATE ===" in content:
            agg = content.split("=== AGGREGATE ===")[1].split("=== Classification Confusion ===")[0]
            print(f"\n--- {PERSONA} ---{agg}")
